# 06 — Data Validation & Quality Checks

**Goal:** Prove the data is trustworthy before sharing the dashboard.

**Rule:** ALL checks must pass before the dashboard goes live.

**Non-technical summary:** This notebook is like a pre-flight checklist. If any item fails, we fix it before showing stakeholders anything.

---


In [ ]:
from sqlalchemy import text
import sys
sys.path.insert(0, '..')
from src.utils.db_loader import get_engine

engine = get_engine()
passed = 0
failed = 0

def check(name, result, expected=True):
    global passed, failed
    ok = result == expected
    if ok: passed += 1
    else:  failed += 1
    print(f'{'✅ PASS' if ok else '❌ FAIL'} — {name}')

## Running All Checks

In [ ]:
with engine.connect() as conn:
    neg = conn.execute(text('SELECT COUNT(*) FROM fact_sales WHERE revenue < 0')).scalar()
    check('No negative revenue', neg == 0)

    null_d = conn.execute(text('SELECT COUNT(*) FROM fact_sales WHERE order_date IS NULL')).scalar()
    check('No NULL order dates', null_d == 0)

    dups = conn.execute(text('SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM fact_sales')).scalar()
    check('No duplicate order IDs', dups == 0)

    rfm_n = conn.execute(text('SELECT COUNT(*) FROM ml_rfm_segments')).scalar()
    check('RFM table populated', rfm_n > 0)

    anom_n = conn.execute(text('SELECT COUNT(*) FROM ml_anomaly_scores')).scalar()
    check('Anomaly scores populated', anom_n > 0)

    fc_n = conn.execute(text('SELECT COUNT(*) FROM revenue_forecast')).scalar()
    check('Forecast table populated', fc_n > 0)

print(f'\n📊 {passed} passed | {failed} failed')
if failed:
    raise Exception(f'{failed} checks failed — DO NOT publish dashboard')
else:
    print('\n✅ All checks passed. Dashboard is safe to share.')